In [1]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

Starting virtual X frame buffer: Xvfb.


In [2]:
%pip install gymnasium[atari]

Note: you may need to restart the kernel to use updated packages.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.




In [2]:
%pip install "gymnasium[atari]"

Note: you may need to restart the kernel to use updated packages.


In [5]:
import atari_wrappers
from importlib import reload
reload(atari_wrappers)

<module 'atari_wrappers' from '/notebooks/week06_policy_based/atari_wrappers.py'>

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import gymnasium as gym
import ale_py
from atari_wrappers import nature_dqn_env
import psutil
gym.register_envs(ale_py)


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = psutil.cpu_count()  # change this if you have more than 8 CPU ;)
print("Number of CPU: ", nenvs)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32

Number of CPU:  32


A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [6]:
env.action_space.n

np.int64(6)

In [5]:
# import tensorflow as torch
# import torch as tf
import torch
import torch.nn.init

def init_orthogonal(m):
    if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear)):
        torch.nn.init.orthogonal_(m.weight, gain=2**0.5)
        torch.nn.init.zeros_(m.bias)

class ConvBackbone(torch.nn.Sequential):
    def __init__(self, c_in = 4, output_size=512):
        layers = [torch.nn.Conv2d(c_in, 32, 8, 4),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, 4, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, 1),
            torch.nn.ReLU(),
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 7 * 7, output_size), # filters * ((((inp_size / stride1) - 1) / stride2) - 1)
            torch.nn.ReLU()
                 ]
        for layer in layers:
            init_orthogonal(layer)
        super().__init__(*layers)

class DoubleHead(torch.nn.Module):
    def __init__(self, n_actions, inp_size=512):
        super().__init__()
        self._value_stream = torch.nn.Linear(inp_size, 1)
        init_orthogonal(self._value_stream)
        self._advantage_stream = torch.nn.Linear(inp_size, n_actions)
        init_orthogonal(self._advantage_stream)

    def forward(self, x: torch.Tensor):
        value = self._value_stream(x)
        advantage = self._advantage_stream(x)
        return (value, advantage)


class DeepNet(torch.nn.Sequential):
    def __init__(self, n_actions):
        conv_backbone = ConvBackbone()
        double_head = DoubleHead(n_actions)
        super().__init__(conv_backbone, double_head)


model = DeepNet(env.action_space.n)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [66]:
from torch.distributions import Categorical

class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        values, logits = self.model(torch.tensor(inputs, dtype=torch.float32))
        dist = Categorical(logits=logits)
        actions = dist.sample()
        log_probs = dist.log_prob(actions)
        return {"actions": actions.data.cpu().numpy(), "logits": logits, "log_probs": log_probs, "values": values.squeeze()}

In [67]:
### Test policy
policy = Policy(model)
policy.act(obs)

{'actions': array([0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 1, 1, 1, 0, 0, 1, 1, 1]),
 'logits': tensor([[-0.1488, -0.0725],
         [-0.0221, -0.2503],
         [-0.1269, -0.0268],
         [-0.0324, -0.2069],
         [-0.1034, -0.0911],
         [-0.0753,  0.2302],
         [ 0.1304, -0.4925],
         [-0.0915, -0.1164],
         [-0.0904, -0.1795],
         [-0.1462, -0.1175],
         [ 0.0051, -0.3469],
         [-0.0583, -0.2290],
         [-0.1425,  0.0674],
         [-0.1497,  0.0021],
         [-0.1229, -0.0636],
         [-0.1391, -0.1469],
         [-0.0666, -0.1808],
         [-0.0393, -0.2077],
         [ 0.3430, -1.4098],
         [-0.1348,  0.0805],
         [-0.0927, -0.1755],
         [ 0.0746, -0.3616],
         [-0.1229, -0.0910],
         [ 0.0426, -0.3268],
         [-0.0663, -0.1495],
         [-0.1686,  0.0767],
         [-0.0386, -0.2483],
         [ 0.0374, -0.3215],
         [-0.0660, -0.1603],
         [ 0.0340, -0.36

Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [8]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [91]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.
        latest_obs = trajectory['state']['latest_observation']
        latest_act = self.policy.act(latest_obs)
        latest_val = latest_act['values'].detach().cpu().numpy()
        trajectory['value_targets'] = self.calculate_value_targets(trajectory, latest_val)

    def calculate_value_targets(self, trajectory, latest_val):
        value_targets = [torch.zeros(32, 1) for _ in range(len(trajectory['rewards']))]
        i = len(trajectory['rewards']) - 1
        for reward in reversed(trajectory['rewards']):
            latest_val = np.where(trajectory['resets'][i], reward, reward + self.gamma * latest_val)
            value_targets[i] = torch.tensor(latest_val, dtype=torch.float32)
            i -= 1
        return value_targets
        

In [10]:
# Testing computing value targets
rewards = [np.array([1] * 32), np.array([1] * 32), np.array([1] * 32), np.array([1] * 32)]
resets = [np.array([False] * 32), np.array([False] * 32), np.array([True] * 32), np.array([False] * 32)]
latest_val = np.array([16] * 32)
resets[2][5] = False
resets[2][-1] = False
trajectory = {"rewards": rewards, "resets": resets}

cvt = ComputeValueTargets(None, 0.5)
value_targets = cvt.calculate_value_targets(trajectory, latest_val)
print(value_targets)

[tensor([1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 2.8750, 1.7500, 1.7500, 1.7500,
        1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500,
        1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500, 1.7500,
        1.7500, 1.7500, 1.7500, 1.7500, 2.8750]), tensor([1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 3.7500, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000, 1.5000,
        1.5000, 1.5000, 1.5000, 1.5000, 3.7500]), tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 5.5000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 5.5000]), tensor([9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9.,
        9., 9., 9., 9., 9., 9., 9., 9., 9., 9., 9

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [73]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.
        for key in trajectory.keys():
            value = trajectory[key]
            if isinstance(value, list) and isinstance(value[0], torch.Tensor):
                trajectory[key] = torch.cat(value, dim=0)
            elif isinstance(value, list) and isinstance(value[0], np.ndarray):
                trajectory[key] = np.concatenate(value, axis=0)

In [12]:
model = DeepNet(env.action_space.n)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


In [13]:
example_traj = runner.get_next()
advantage = example_traj['value_targets'] - example_traj['values']
print(example_traj['values'], example_traj['value_targets'], example_traj['log_probs'])
print(advantage)
print(example_traj['log_probs'] * advantage)

tensor([0.2365, 0.2365, 0.2346, 0.2365, 0.2346, 0.2346, 0.2365, 0.2346, 0.2365,
        0.2365, 0.2365, 0.2365, 0.2346, 0.2365, 0.2365, 0.2365, 0.2346, 0.2346,
        0.2365, 0.2346, 0.2365, 0.2365, 0.2365, 0.2346, 0.2346, 0.2365, 0.2346,
        0.2365, 0.2346, 0.2346, 0.2365, 0.2346, 0.2365, 0.2365, 0.2325, 0.2362,
        0.2346, 0.2346, 0.2365, 0.2346, 0.2365, 0.2362, 0.2362, 0.2362, 0.2346,
        0.2365, 0.2362, 0.2362, 0.2346, 0.2346, 0.2365, 0.2346, 0.2365, 0.2365,
        0.2362, 0.2346, 0.2346, 0.2362, 0.2346, 0.2365, 0.2325, 0.2325, 0.2365,
        0.2346, 0.2362, 0.2362, 0.2312, 0.2317, 0.2325, 0.2346, 0.2362, 0.2325,
        0.2362, 0.2317, 0.2317, 0.2317, 0.2325, 0.2362, 0.2317, 0.2317, 0.2325,
        0.2346, 0.2362, 0.2325, 0.2362, 0.2362, 0.2317, 0.2325, 0.2325, 0.2317,
        0.2325, 0.2362, 0.2312, 0.2312, 0.2362, 0.2325, 0.2317, 0.2317, 0.2373,
        0.2298, 0.2364, 0.2325, 0.2317, 0.2312, 0.2317, 0.2298, 0.2287, 0.2287,
        0.2312, 0.2317, 0.2287, 0.2298, 

Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [130]:
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import time

class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.writer = SummaryWriter(f"logs/{env_name}")

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        dist = Categorical(logits=trajectory['logits'])
        entropy = dist.entropy()
        advantage = trajectory['value_targets'] - trajectory['values']
        result = trajectory['log_probs'] * advantage.detach()
        mean_entropy = torch.mean(entropy)
        result = -1 * torch.mean(result) - self.entropy_coef * mean_entropy
        self.add_summary("Entropy", mean_entropy)
        self.add_summary("Policy loss", result)
        self.add_summary("Value targets", torch.mean(trajectory['value_targets']))
        self.add_summary("Value predictions", torch.mean(trajectory['values']))
        self.add_summary("Advantage", torch.mean(advantage))
        return result

    def value_loss(self, trajectory):
        value_loss = F.mse_loss(trajectory['value_targets'].detach(), trajectory['values'])
        res_sum = torch.sum(torch.square(trajectory['value_targets'] - trajectory['values']))
        # l = len(trajectory['value_targets'])
        # result = total_sum / l
        self.add_summary("Value loss", value_loss)
        value_mean = torch.mean(trajectory['value_targets'])
        total_sum = torch.sum(torch.square(trajectory['value_targets'] - value_mean))
        det_coeff = 1 - (res_sum / total_sum)
        self.add_summary("Coefficient of determination", det_coeff)
        return value_loss 

    def loss(self, trajectory):
        # pdb.set_trace()
        result = self.policy_loss(trajectory) + self.value_loss_coef * self.value_loss(trajectory)
        self.add_summary("Loss", result)
        return result

    def step(self, trajectory, step_var):
        self.step_var = step_var
        self.optimizer.zero_grad()
        loss = self.loss(trajectory)
        loss.backward()
        grad_norm = self.calculate_grad_norm()
        self.add_summary("Gradient norm", grad_norm)
        torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), max_norm=self.max_grad_norm)
        self.optimizer.step()

    def calculate_grad_norm(self):
        total_grad_norm = 0
        for p in self.policy.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2).item()
                total_grad_norm += param_norm ** 2
        return total_grad_norm ** 0.5

    def add_summary(self, name, value):
        if isinstance(value, dict):
            self.writer.add_scalars(name, value, self.step_var)
        else:
            self.writer.add_scalar(name, value, self.step_var)

In [104]:
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import time

class ComputeValueTargetsReinforce:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.
        value_targets = [torch.zeros(32, 1) for _ in range(len(trajectory['rewards']))]
        i = len(trajectory['rewards']) - 1
        latest_val = 0
        for reward in reversed(trajectory['rewards']):
            latest_val = np.where(trajectory['resets'][i], reward, reward + self.gamma * latest_val)
            value_targets[i] = torch.tensor(latest_val, dtype=torch.float32)
            i -= 1
        trajectory['value_targets'] = value_targets
        

class PolicyReinforce:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        values, logits = self.model(torch.tensor(inputs, dtype=torch.float32))
        probs = torch.nn.functional.softmax(logits, dim=-1)
        log_probs = torch.nn.functional.log_softmax(logits, -1)
        actions = torch.multinomial(
            probs, 
            num_samples=1, 
            replacement=True
        ).squeeze(1)
        return {"actions": actions.detach().cpu().numpy(), "logits": logits, "log_probs": log_probs, "probs": probs}

class REINFORCE:
    def __init__(self,
                 policy,
                 optimizer,
                 entropy_coef=0.01):
        self.policy = policy
        self.optimizer = optimizer
        self.entropy_coef = entropy_coef
        self.writer = SummaryWriter(f"logs/{env_name}")

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        actions = torch.tensor(trajectory['actions'], dtype=torch.int64)
        log_probs = trajectory['log_probs']
        probs = trajectory['probs']
        log_probs_for_actions = torch.sum(
            log_probs * F.one_hot(actions, n_actions), dim=1)
    
        probs_for_actions = torch.sum(probs * F.one_hot(actions, n_actions), dim=1)

        entropy = -1 * torch.mean(log_probs_for_actions * probs_for_actions)
        cumulative_returns = trajectory['value_targets']
        result = torch.mean(log_probs_for_actions * cumulative_returns.detach())
        result = -1 * result + self.entropy_coef * entropy
        self.add_summary("Entropy", entropy)
        self.add_summary("Value targets", torch.mean(trajectory['value_targets']))

        print("actions", actions.shape)
        print("cumulative_returns", cumulative_returns.shape)
        print("logits", trajectory['logits'].shape)
        print("probs", probs.shape)
        print("log_probs", log_probs.shape)
        print("probs_for_actions", probs_for_actions.shape)
        print("log_probs_for_actions", log_probs_for_actions.shape)

        return result

    def loss(self, trajectory):
        # pdb.set_trace()
        result = self.policy_loss(trajectory)
        self.add_summary("Loss", result)
        return result

    def step(self, trajectory, step_var):
        self.step_var = step_var
        self.optimizer.zero_grad()
        loss = self.loss(trajectory)
        loss.backward()
        self.optimizer.step()

    def add_summary(self, name, value):
        if isinstance(value, dict):
            self.writer.add_scalars(name, value, self.step_var)
        else:
            self.writer.add_scalar(name, value, self.step_var)

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.





In [105]:
from atari_wrappers import cart_pole_env
env_name = "CartPole-v1"
env = cart_pole_env(env_name, nenvs=nenvs, summaries=summaries, clip_reward=False)
obs, _ = env.reset()
n_actions = env.action_space.n
state_dim = env.observation_space.shape
print(n_actions, state_dim)

class DualHeadNet(torch.nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self._value_stream = torch.nn.Linear(30, 1)
        self._advantage_stream = torch.nn.Linear(30, n_actions)

    def forward(self, inp):
        value = self._value_stream(inp)
        logits = self._advantage_stream(inp)
        return (value, logits)


model = torch.nn.Sequential(
  torch.nn.Linear(state_dim[0], 30),
  torch.nn.ReLU(),
  DualHeadNet(n_actions)
)


policy = PolicyReinforce(model)

total_steps = int(300) # 3000
init_epsilon = 7e-4
final_epsilon = 0
smoothing_constant = 0.99
eps = 1e-5
step = 0

reinforce = REINFORCE(policy, torch.optim.Adam(model.parameters(), 1e-3))
          # torch.optim.RMSprop(policy.model.parameters(), init_epsilon, smoothing_constant, eps))
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=1000,
    transforms=[
        ComputeValueTargetsReinforce(policy),
        MergeTimeBatch(),
    ],
)



# runner.get_next()

2 (4,)


Process Process-968:
Process Process-967:
Process Process-965:
Process Process-992:
Process Process-990:
Process Process-991:
Process Process-964:
Process Process-969:
Process Process-989:
Process Process-977:
Process Process-984:
Process Process-983:
Process Process-985:
Process Process-986:
Process Process-987:
Process Process-981:
Process Process-962:
Process Process-963:
Process Process-972:
Process Process-971:
Process Process-988:
Process Process-976:
Process Process-970:
Process Process-982:
Process Process-979:
Process Process-973:
Process Process-980:
Process Process-975:
Process Process-974:
Process Process-966:
Process Process-961:
Process Process-978:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
T

In [103]:
step = 0
with trange(step, total_steps + 1) as progress_bar:
    for step in progress_bar:
        if not is_enough_ram():
            print('Less than 100 MB RAM available, freezing.')
            print('Ensure everything is okay and use KeyboardInterrupt to continue.')
            wait_for_keyboard_interrupt()
        # a2c.optimizer.lr = linear_decay(init_epsilon, final_epsilon, step, total_steps)
        # print(a2c.optimizer.lr)
        trajectory = runner.get_next()
        reinforce.step(trajectory, step)

  0%|          | 0/301 [00:00<?, ?it/s]

In [106]:
trajectory = runner.get_next()
reinforce.step(trajectory, step)

actions torch.Size([32000])
cumulative_returns torch.Size([32000])
logits torch.Size([32000, 2])
probs torch.Size([32000, 2])
log_probs torch.Size([32000, 2])
probs_for_actions torch.Size([32000])
log_probs_for_actions torch.Size([32000])


In [116]:
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import time

class ReinforceA2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.writer = SummaryWriter(f"logs/{env_name}")

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        dist = Categorical(logits=trajectory['logits'])
        entropy = dist.entropy()
        advantage = trajectory['value_targets'] #- trajectory['values']
        result = trajectory['log_probs'] * advantage.detach()
        mean_entropy = torch.mean(entropy)
        result = -1 * torch.mean(result) - self.entropy_coef * mean_entropy
        self.add_summary("Entropy", mean_entropy)
        self.add_summary("Policy loss", result)
        self.add_summary("Value targets", torch.mean(trajectory['value_targets']))
        self.add_summary("Value predictions", torch.mean(trajectory['values']))
        self.add_summary("Advantage", torch.mean(advantage))
        # print("value_targets", trajectory['value_targets'].shape)
        # print("values", trajectory['values'].shape)
        # print("logits", trajectory['logits'].shape)
        # print("log_probs", trajectory['log_probs'].shape)
        return result

    def value_loss(self, trajectory):
        value_loss = F.mse_loss(trajectory['value_targets'].detach(), trajectory['values'])
        total_sum = torch.sum(torch.square(trajectory['value_targets'] - trajectory['values']))
        # l = len(trajectory['value_targets'])
        # result = total_sum / l
        self.add_summary("Value loss", value_loss)
        value_mean = torch.mean(trajectory['value_targets'])
        res_sum = torch.sum(torch.square(trajectory['value_targets'] - value_mean))
        det_coeff = 1 - (res_sum / total_sum)
        self.add_summary("Coefficient of determination", det_coeff)
        return value_loss 

    def loss(self, trajectory):
        # pdb.set_trace()
        result = self.policy_loss(trajectory) #+ self.value_loss_coef * self.value_loss(trajectory)
        self.add_summary("Loss", result)
        return result

    def step(self, trajectory, step_var):
        self.step_var = step_var
        self.optimizer.zero_grad()
        loss = self.loss(trajectory)
        loss.backward()
        grad_norm = self.calculate_grad_norm()
        self.add_summary("Gradient norm", grad_norm)
        torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), max_norm=self.max_grad_norm)
        self.optimizer.step()

    def calculate_grad_norm(self):
        total_grad_norm = 0
        for p in self.policy.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2).item()
                total_grad_norm += param_norm ** 2
        return total_grad_norm ** 0.5

    def add_summary(self, name, value):
        if isinstance(value, dict):
            self.writer.add_scalars(name, value, self.step_var)
        else:
            self.writer.add_scalar(name, value, self.step_var)

In [123]:
from atari_wrappers import cart_pole_env
env_name = "CartPole-v1"
env = cart_pole_env(env_name, nenvs=2, summaries=summaries, clip_reward=False)
obs, _ = env.reset()
n_actions = env.action_space.n
state_dim = env.observation_space.shape
print(n_actions, state_dim)

class DualHeadNet(torch.nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self._value_stream = torch.nn.Linear(30, 1)
        self._advantage_stream = torch.nn.Linear(30, n_actions)

    def forward(self, inp):
        value = self._value_stream(inp)
        logits = self._advantage_stream(inp)
        return (value, logits)


model = torch.nn.Sequential(
  torch.nn.Linear(state_dim[0], 30),
  torch.nn.ReLU(),
  DualHeadNet(n_actions)
)


policy = Policy(model)

total_steps = int(10000) # 3000
init_epsilon = 7e-4
final_epsilon = 0
smoothing_constant = 0.99
eps = 1e-5
step = 0

reinforce_a2c = ReinforceA2C(policy, torch.optim.Adam(model.parameters(), 1e-3))
          # torch.optim.RMSprop(policy.model.parameters(), init_epsilon, smoothing_constant, eps))
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=100,
    transforms=[
        ComputeValueTargetsReinforce(policy),
        MergeTimeBatch(),
    ],
)



# runner.get_next()

2 (4,)


Process Process-1158:
Process Process-1157:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/conda/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/conda/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/conda/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/notebooks/week06_policy_based/env_batch.py", line 149, in worker
    cmd, data = worker_connection.recv()
                ~~~~~~~~~~~~~~~~~~~~~~^^
  File "/notebooks/week06_policy_based/env_batch.py", line 149, in worker
    cmd, data = worker_connection.recv()
                ~~~~~~~~~~~~~~~~~~~~~

In [124]:
step = 0
with trange(step, total_steps + 1) as progress_bar:
    for step in progress_bar:
        if not is_enough_ram():
            print('Less than 100 MB RAM available, freezing.')
            print('Ensure everything is okay and use KeyboardInterrupt to continue.')
            wait_for_keyboard_interrupt()
        # a2c.optimizer.lr = linear_decay(init_epsilon, final_epsilon, step, total_steps)
        # print(a2c.optimizer.lr)
        trajectory = runner.get_next()
        reinforce_a2c.step(trajectory, step)

  0%|          | 0/10001 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [112]:
trajectory = runner.get_next()
reinforce_a2c.step(trajectory, step)

value_targets torch.Size([32000])
values torch.Size([32000])
logits torch.Size([32000, 2])
log_probs torch.Size([32000])


In [131]:
from atari_wrappers import cart_pole_env
env_name = "CartPole-v1"
env = cart_pole_env(env_name, nenvs=nenvs, summaries=summaries, clip_reward=False)
obs, _ = env.reset()
n_actions = env.action_space.n
state_dim = env.observation_space.shape
print(n_actions, state_dim)

class DualHeadNet(torch.nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self._value_stream = torch.nn.Linear(30, 1)
        self._advantage_stream = torch.nn.Linear(30, n_actions)

    def forward(self, inp):
        value = self._value_stream(inp)
        logits = self._advantage_stream(inp)
        return (value, logits)


model = torch.nn.Sequential(
  torch.nn.Linear(state_dim[0], 30),
  torch.nn.ReLU(),
  DualHeadNet(n_actions)
)


policy = Policy(model)

total_steps = int(10000) # 3000
init_epsilon = 7e-4
final_epsilon = 0
smoothing_constant = 0.99
eps = 1e-5
step = 0

a2c = A2C(policy, torch.optim.RMSprop(policy.model.parameters(), init_epsilon, smoothing_constant, eps))
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=100,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)



# runner.get_next()

2 (4,)


Process Process-1178:
Process Process-1165:
Process Process-1163:
Process Process-1192:
Process Process-1176:
Process Process-1187:
Process Process-1175:
Process Process-1174:
Process Process-1189:
Process Process-1190:
Process Process-1188:
Process Process-1167:
Process Process-1186:
Process Process-1177:
Process Process-1182:
Process Process-1185:
Process Process-1164:
Process Process-1193:
Process Process-1194:
Process Process-1181:
Process Process-1191:
Process Process-1184:
Process Process-1183:
Process Process-1171:
Process Process-1180:
Process Process-1173:
Process Process-1166:
Process Process-1170:
Process Process-1179:
Process Process-1168:
Process Process-1169:
Process Process-1172:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most 

In [132]:
step = 0
with trange(step, total_steps + 1) as progress_bar:
    for step in progress_bar:
        if not is_enough_ram():
            print('Less than 100 MB RAM available, freezing.')
            print('Ensure everything is okay and use KeyboardInterrupt to continue.')
            wait_for_keyboard_interrupt()
        # a2c.optimizer.lr = linear_decay(init_epsilon, final_epsilon, step, total_steps)
        # print(a2c.optimizer.lr)
        trajectory = runner.get_next()
        a2c.step(trajectory, step)

  0%|          | 0/10001 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [28]:
%pip install "gymnasium[classic-control]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 38.6 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [32]:
def evaluate_cart_pole(env, policy, n_games=1, t_max=10000, seed=None):
    """ Plays n_games full games.  Returns mean reward. """
    rewards = []
    for _ in range(n_games):
        s, _ = env.reset(seed=seed)
        reward = 0
        for _ in range(t_max):
            # s = s.reshape(1, 4, 84, 84)
            # print("s", s.shape, s)
            act = policy.act(s)
            print(act)
            
            s, r, terminated, truncated, _ = env.step(act["actions"][-1])
            reward += r
            if terminated or truncated:
                break

        rewards.append(reward)
    return np.mean(rewards)

In [33]:
# record sessions
n_lives = 5
from gymnasium.wrappers import RecordVideo

with cart_pole_env(env_name, nenvs=None, summaries=None, clip_reward=False) as env_cartpole, RecordVideo(
    env=env_cartpole, video_folder="./videos_cartpole", episode_trigger=lambda episode_number: True
) as env_monitor:
    sessions = [
        evaluate_cart_pole(env_monitor, a2c.policy, n_games=n_lives) for _ in range(10)
    ]


/opt/conda/lib/python3.13/site-packages/gymnasium/envs/classic_control/cartpole.py:214: UserWarning: WARN: You are calling 'step()' even though this environment has already returned terminated = True. You should always call 'reset()' once you receive 'terminated = True' -- any further steps are undefined behavior.
  logger.warn(
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM defaul

{'actions': array(1), 'logits': tensor([-1.1876,  6.1763], grad_fn=<ViewBackward0>), 'log_probs': tensor(-0.0006, grad_fn=<SqueezeBackward1>), 'values': tensor(80.8617, grad_fn=<SqueezeBackward0>)}


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [121]:
a = [np.array(0), np.array(1), np.array(0), np.array(0), np.array(1)]
np

array([8.405994e+22], dtype=float32)

In [12]:
#if you use TensorboardSummaries
%load_ext tensorboard
# %reload_ext tensorboard
%tensorboard --logdir logs
from torch.utils.tensorboard import SummaryWriter

a2c = A2C(policy, torch.optim.RMSprop(policy.model.parameters(), 7e-4, 0.99, 1e-5))


Reusing TensorBoard on port 6006 (pid 24236), started 6 days, 4:38:43 ago. (Use '!kill 24236' to kill it.)

In [16]:
from tqdm.auto import trange

In [17]:
import time

def wait_for_keyboard_interrupt():
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        pass

In [18]:
def is_enough_ram(min_available_gb=0.1):
    mem = psutil.virtual_memory()
    return mem.available >= min_available_gb * (1024**3)


def linear_decay(
    init_val: float, final_val: float, cur_step: int, total_steps: int
) -> float:
    if cur_step >= total_steps:
        return final_val
    return (init_val * (total_steps - cur_step) + final_val * cur_step) / total_steps

In [134]:
import pdb
env_name = "SpaceInvadersNoFrameskip-v4"
# env_name = "FreewayNoFrameskip-v4"
total_steps = int(10 * 10**6 / (5*nenvs)) # 3000
init_epsilon = 7e-4
final_epsilon = 0
smoothing_constant = 0.99
eps = 1e-5
step = 0

In [135]:
env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
model = DeepNet(env.action_space.n)
policy = Policy(model)
a2c = A2C(policy, torch.optim.RMSprop(policy.model.parameters(), init_epsilon, smoothing_constant, eps))
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]
A.L.E: Arcade Learning Environment (vers

In [ ]:
step = 0
with trange(step, total_steps + 1) as progress_bar:
    for step in progress_bar:
        if not is_enough_ram():
            print('Less than 100 MB RAM available, freezing.')
            print('Ensure everything is okay and use KeyboardInterrupt to continue.')
            wait_for_keyboard_interrupt()
        a2c.optimizer.lr = linear_decay(init_epsilon, final_epsilon, step, total_steps)
        trajectory = runner.get_next()
        a2c.step(trajectory, step)

  0%|          | 0/62501 [00:00<?, ?it/s]

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [26]:
def evaluate(env, policy, n_games=1, t_max=10000, seed=None):
    """ Plays n_games full games.  Returns mean reward. """
    rewards = []
    for _ in range(n_games):
        s, _ = env.reset(seed=seed)
        reward = 0
        for _ in range(t_max):
            s = s.reshape(1, 4, 84, 84)
            # print("s", s.shape, s)
            act = policy.act(s)
            
            s, r, terminated, truncated, _ = env.step(act["actions"][-1])
            reward += r
            if terminated or truncated:
                break

        rewards.append(reward)
    return np.mean(rewards)

In [317]:
# record sessions
n_lives = 5
from gymnasium.wrappers import RecordVideo

with nature_dqn_env(env_name, nenvs=None, summaries=None) as env_fw, RecordVideo(
    env=env_fw, video_folder="./videos_fw", episode_trigger=lambda episode_number: True
) as env_monitor:
    sessions = [
        evaluate(env_monitor, a2c.policy, n_games=n_lives) for _ in range(10)
    ]


In [320]:
# Show video. This may not work in some setups. If it doesn't
# work for you, you can download the videos and view them locally.

from pathlib import Path
from base64 import b64encode
from IPython.display import HTML

video_paths = sorted([s for s in Path('videos_fw').iterdir() if s.suffix == '.mp4'])
video_path = video_paths[3]  # You can also try other indices

if 'google.colab' in sys.modules:
    # https://stackoverflow.com/a/57378660/1214547
    with video_path.open('rb') as fp:
        mp4 = fp.read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
else:
    data_url = str(video_path)

HTML("""
<video width="640" height="480" controls>
  <source src="{}" type="video/mp4">
</video>
""".format(data_url))

In [29]:
traj = runner.get_next()

actions [array([5, 5, 0, 4, 5, 0, 2, 2, 2, 0, 1, 2, 0, 5, 0, 0, 4, 0, 3, 3, 0, 4,
       2, 5, 5, 2, 2, 4, 1, 2, 1, 3]), array([5, 1, 5, 1, 5, 5, 5, 4, 2, 0, 0, 1, 0, 3, 2, 3, 0, 2, 2, 4, 2, 1,
       2, 5, 2, 1, 2, 0, 5, 0, 5, 2]), array([4, 0, 5, 1, 0, 3, 2, 5, 0, 5, 0, 4, 1, 0, 2, 0, 2, 0, 0, 3, 0, 2,
       3, 1, 4, 4, 4, 0, 5, 1, 2, 1]), array([3, 0, 5, 2, 3, 5, 5, 2, 5, 2, 2, 1, 4, 0, 0, 5, 3, 3, 3, 2, 3, 0,
       0, 2, 4, 0, 0, 1, 2, 4, 2, 1]), array([1, 0, 0, 0, 3, 2, 2, 1, 2, 1, 5, 5, 0, 1, 2, 0, 5, 4, 1, 0, 3, 3,
       0, 1, 1, 2, 4, 4, 4, 0, 2, 1])]
[5 5 0 4 5 0 2 2 2 0 1 2 0 5 0 0 4 0 3 3 0 4 2 5 5 2 2 4 1 2 1 3 5 1 5 1 5
 5 5 4 2 0 0 1 0 3 2 3 0 2 2 4 2 1 2 5 2 1 2 0 5 0 5 2 4 0 5 1 0 3 2 5 0 5
 0 4 1 0 2 0 2 0 0 3 0 2 3 1 4 4 4 0 5 1 2 1 3 0 5 2 3 5 5 2 5 2 2 1 4 0 0
 5 3 3 3 2 3 0 0 2 4 0 0 1 2 4 2 1 1 0 0 0 3 2 2 1 2 1 5 5 0 1 2 0 5 4 1 0
 3 3 0 1 1 2 4 4 4 0 2 1]
logits [tensor([[-0.0901, -0.0841,  0.3383, -0.2612, -0.3009, -0.0752],
        [-0.0688, -0.1443,  0.134

In [23]:
torch.cat(torch.Tensor([[3,4,5],[0,9,8]]), dim=0)

TypeError: cat() received an invalid combination of arguments - got (Tensor, dim=int), but expected one of:
 * (tuple of Tensors tensors, int dim = 0, *, Tensor out = None)
 * (tuple of Tensors tensors, name dim, *, Tensor out = None)


In [286]:
env.action_space.n

np.int64(3)

In [314]:
e = nature_dqn_env(env_name, nenvs=None, summaries=None)
s, _ = e.reset()
s = s.reshape(1,4,84,84)
print(s.shape)
m = torch.nn.Sequential(torch.nn.Conv2d(4, 32, 8, 4), torch.nn.ReLU(), torch.nn.Conv2d(32, 64, 4, 2),
            torch.nn.ReLU(),
            torch.nn.Conv2d(64, 64, 3, 1),
            torch.nn.ReLU(),
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 7 * 7, 512) # filters * ((((inp_size / stride1) - 1) / stride2) - 1)
            )
m(torch.Tensor(s)).shape
m2 = DeepNet(3)
m2(torch.Tensor(s))

(1, 4, 84, 84)


(tensor([[0.4305]], grad_fn=<AddmmBackward0>),
 tensor([[0.9396, 0.0952, 0.4003]], grad_fn=<AddmmBackward0>))

In [310]:
env2 = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
model2 = DeepNet(env.action_space.n)
policy2 = Policy(model2)
a2c2 = A2C(policy2, torch.optim.RMSprop(policy2.model.parameters(), init_epsilon, smoothing_constant, eps))
runner2 = EnvRunner(
    env=env2,
    policy=policy2,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy2),
        MergeTimeBatch(),
    ],
)

Process Process-2092:
Process Process-2095:
Process Process-2086:
Process Process-2099:
Process Process-2110:
Process Process-2091:
Process Process-2089:
Process Process-2105:
Process Process-2090:
Process Process-2109:
Process Process-2088:
Process Process-2106:
Process Process-2082:
Process Process-2087:
Process Process-2108:
Process Process-2101:
Process Process-2102:
Process Process-2100:
Process Process-2085:
Process Process-2098:
Process Process-2104:
Process Process-2083:
Process Process-2094:
Process Process-2112:
Process Process-2111:
Process Process-2107:
Process Process-2096:
Process Process-2097:
Process Process-2103:
Process Process-2084:
Process Process-2093:
Process Process-2081:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most 

In [311]:
with trange(step, total_steps + 1) as progress_bar:
    for step in progress_bar:
        if not is_enough_ram():
            print('Less than 100 MB RAM available, freezing.')
            print('Ensure everything is okay and use KeyboardInterrupt to continue.')
            wait_for_keyboard_interrupt()
        a2c2.optimizer.lr = linear_decay(init_epsilon, final_epsilon, step, total_steps)
        trajectory2 = runner2.get_next()
        a2c2.step(trajectory2, step * nenvs)

  0%|          | 0/3001 [00:00<?, ?it/s]

(32, 4, 84, 84)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x3136 and 49x512)

In [322]:
cartpole_env = gym.make("CartPole-v1")

In [323]:
cps, _ = cartpole_env.reset()
cps

array([ 0.01381216, -0.01531388, -0.0275696 , -0.01983042], dtype=float32)

In [277]:
torch.cuda.is_available()

True

In [30]:
ob = env.reset()[0]

In [32]:
ob.shape

(32, 4, 84, 84)

In [34]:
policy.model(torch.Tensor(ob))

(tensor([[0.1213],
         [0.1213],
         [0.1170],
         [0.1213],
         [0.1170],
         [0.1170],
         [0.1213],
         [0.1170],
         [0.1213],
         [0.1213],
         [0.1213],
         [0.1213],
         [0.1170],
         [0.1213],
         [0.1213],
         [0.1213],
         [0.1170],
         [0.1170],
         [0.1213],
         [0.1170],
         [0.1213],
         [0.1213],
         [0.1213],
         [0.1170],
         [0.1170],
         [0.1213],
         [0.1170],
         [0.1213],
         [0.1170],
         [0.1170],
         [0.1213],
         [0.1170]], grad_fn=<AddmmBackward0>),
 tensor([[-0.2054,  0.1930, -0.1048,  0.2083,  0.0584,  0.1301],
         [-0.2054,  0.1930, -0.1048,  0.2083,  0.0584,  0.1301],
         [-0.2043,  0.1917, -0.1091,  0.1859,  0.0523,  0.1349],
         [-0.2054,  0.1930, -0.1048,  0.2083,  0.0584,  0.1301],
         [-0.2043,  0.1917, -0.1091,  0.1859,  0.0523,  0.1349],
         [-0.2043,  0.1917, -0.1091,  0

In [19]:
from torch.utils.tensorboard import SummaryWriter
import time

logdir = "logs/test_run"

writer = SummaryWriter(log_dir=logdir)
for step in range(10):
    writer.add_scalar("dummy/value", step, step)
    time.sleep(0.1)
writer.close()

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.